# Entrenamiento de VotoIA — ResNet50

**Práctica de clasificación de imágenes.** Esta adaptación sigue los notebooks del profesor de **Clasificación de Imágenes** y **ResNets, Global Average Pooling y Class Activation Maps**.

Utilizamos `ResNet50(weights="imagenet")`, `preprocess_input`, `model.summary`, `model.predict` y `GlobalAveragePooling2D`. Agregamos capas de clasificación para nuestras **cuatro clases de marcas manuscritas**.

> **Importante:** aquí no hay resultados inventados. Primero debemos recolectar 500–1000 imágenes manuales por categoría con `main.py`. El notebook mostrará métricas reales solo después del entrenamiento. No se fabrican dibujos para llenar el dataset.

## 1. Librerías

En Colab se recomienda habilitar GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*.

In [ ]:
# Librerías similares a las de clase
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.utils import plot_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

## 2. Subir los dibujos reales

En tu PC, abre la carpeta del proyecto. Comprime manualmente la carpeta `dataset` como `dataset.zip` (clic derecho → Comprimir en archivo ZIP). Debe incluir `dataset/aspa_valida/`, `dataset/cruz_valida/`, `dataset/interseccion_fuera/` y `dataset/marca_diferente/`, con sus PNG. Sube solo ese ZIP aquí; no subas el ZIP del código del proyecto.


In [ ]:
# Selecciona el ZIP que comprimiste manualmente en tu PC.
from google.colab import files
import shutil
subidos = files.upload()
archivo_zip = next((name for name in subidos if name.lower().endswith('.zip')), None)
if archivo_zip is None:
    raise ValueError('Debes subir el ZIP de tus dibujos.')
carpeta = Path('/content/votoia_datos')
# Evita mezclar los datos de distintas cargas del notebook.
if carpeta.exists():
    shutil.rmtree(carpeta)
carpeta.mkdir()
with zipfile.ZipFile(archivo_zip) as zf:
    for nombre in zf.namelist():
        destino = (carpeta / nombre).resolve()
        if not destino.is_relative_to(carpeta.resolve()):
            raise ValueError('El ZIP contiene una ruta no válida.')
    zf.extractall(carpeta)
# Admite comprimir dataset/ o seleccionar directamente sus cuatro carpetas.
dataset_dir = carpeta / 'dataset' if (carpeta / 'dataset').is_dir() else carpeta
esperadas = ['aspa_valida', 'cruz_valida', 'interseccion_fuera', 'marca_diferente']
if not all((dataset_dir / clase).is_dir() for clase in esperadas):
    raise ValueError('El ZIP debe contener dataset/ con las cuatro categorías, o las cuatro carpetas directamente.')
print('Carpeta:', dataset_dir)


## 3. Leer etiquetas y contar imágenes

Los dibujos se guardan como PNG dentro de una carpeta por categoría, igual que las carpetas `I`, `N`, `U` del código original. A diferencia del original, el canvas incluye un marco fijo para que la red pueda aprender si una marca cruza dentro o fuera de él.

In [ ]:
# Solo cuatro categorías; el voto en blanco se detecta sin red neuronal.
clases = ['aspa_valida', 'cruz_valida', 'interseccion_fuera', 'marca_diferente']
archivos = []
etiquetas = []
for indice, clase in enumerate(clases):
    imagenes = sorted((dataset_dir / clase).glob('*.png'))
    print(clase, ':', len(imagenes), 'dibujos')
    if len(imagenes) < 500:
        print('  AVISO: aún no se alcanzan los 500 dibujos solicitados por categoría.')
    for path in imagenes:
        archivos.append(str(path))
        etiquetas.append(indice)
if not archivos:
    raise ValueError('No hay imágenes. Revisa las carpetas del ZIP que comprimiste.')
if any(sum(np.array(etiquetas) == i) < 5 for i in range(len(clases))):
    raise ValueError('Para continuar recoge más ejemplos de todas las categorías.')
print('Total de dibujos:', len(archivos))

plt.figure(figsize=(7, 3))
plt.bar(clases, [etiquetas.count(i) for i in range(4)])
plt.xticks(rotation=15)
plt.title('Cantidad de dibujos por categoría')
plt.tight_layout()
plt.show()

## 4. Visualizar unos dibujos del dataset

In [ ]:
from tensorflow.keras.preprocessing import image

plt.figure(figsize=(10, 3))
for i, clase in enumerate(clases):
    archivo = sorted((dataset_dir / clase).glob('*.png'))[0]
    img = image.load_img(archivo, target_size=(200, 200))
    plt.subplot(1, 4, i + 1)
    plt.imshow(img)
    plt.title(clase)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 5. Separar entrenamiento, validación y prueba

Usamos 70% para entrenamiento, 15% para validación y 15% para prueba, con una semilla fija. El conjunto de prueba no se usa durante el aprendizaje.

**Limitación:** si una misma persona dibuja para ambos conjuntos, la prueba puede resultar optimista; sería mejor recoger dibujos de personas distintas para la evaluación final.

In [ ]:
# Conservamos etiquetas enteras para la clasificación multiclase.
archivos = np.array(archivos)
etiquetas = np.array(etiquetas)
x_train, x_resto, y_train, y_resto = train_test_split(
    archivos, etiquetas, test_size=0.30, random_state=42, stratify=etiquetas)
x_val, x_test, y_val, y_test = train_test_split(
    x_resto, y_resto, test_size=0.50, random_state=42, stratify=y_resto)
print('Entrenamiento:', len(x_train))
print('Validación:', len(x_val))
print('Prueba:', len(x_test))

## 6. Preprocesamiento de ResNet50

El profesor utilizó `image.load_img`, `img_to_array`, `np.expand_dims` y `preprocess_input`. Hacemos lo mismo, pero con un lote de imágenes usando `tf.data` para no cargar miles de imágenes RGB en la memoria simultáneamente.

El canvas original mide 200×200. ResNet50 recibe imágenes RGB de **224×224**.

In [ ]:
target_size = (224, 224)
batch_size = 16

def leer_imagen(ruta, etiqueta):
    contenido = tf.io.read_file(ruta)
    imagen = tf.io.decode_png(contenido, channels=3)
    imagen = tf.image.resize(imagen, target_size)
    imagen = tf.cast(imagen, tf.float32)
    imagen = preprocess_input(imagen)    # del notebook del profesor
    return imagen, etiqueta

def crear_dataset(x, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(x), seed=42)
    ds = ds.map(leer_imagen, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = crear_dataset(x_train, y_train, shuffle=True)
val_ds = crear_dataset(x_val, y_val)
test_ds = crear_dataset(x_test, y_test)

## 7. Cargar ResNet50 preentrenada y agregar nuestro clasificador

En clase se usó la red preentrenada con mil categorías de ImageNet. Aquí retiramos esa última clasificación (`include_top=False`), **congelamos la base** y agregamos Global Average Pooling y dos capas `Dense` para nuestras cuatro categorías.

In [ ]:
# Basado en las llamadas utilizadas en los notebooks del profesor.
base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False

model = Sequential([
    base,
    GlobalAveragePooling2D(),
    Dense(64, activation='relu'),
    Dense(len(clases), activation='softmax')
])
model.build((None, 224, 224, 3))
model.summary()

# Como en clase: plot_model (puede requerir Graphviz/pydot en Colab).
try:
    display(plot_model(model, show_shapes=True))
except Exception as exc:
    print('No se pudo dibujar el esquema del modelo:', str(exc))

## 8. Compilar y entrenar

Pérdida `sparse_categorical_crossentropy` porque las etiquetas son números del 0 al 3. Se conserva la base congelada (transfer learning).

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

carpeta_resultados = Path('/content/resultados_entrenamiento')
carpeta_resultados.mkdir(exist_ok=True)
early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                          restore_best_weights=True)

historia = model.fit(train_ds, validation_data=val_ds,
                     epochs=12, callbacks=[early])

## 9. Graficar el entrenamiento

Estas curvas se producen al ejecutar `model.fit`. No hay valores precargados.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(historia.history['accuracy'], label='Entrenamiento')
axes[0].plot(historia.history['val_accuracy'], label='Validación')
axes[0].set(title='Accuracy por época', xlabel='Época', ylabel='Accuracy')
axes[0].legend()
axes[1].plot(historia.history['loss'], label='Entrenamiento')
axes[1].plot(historia.history['val_loss'], label='Validación')
axes[1].set(title='Pérdida por época', xlabel='Época', ylabel='Loss')
axes[1].legend()
plt.tight_layout()
plt.savefig(carpeta_resultados / 'curvas.png', dpi=150)
plt.show()
pd.DataFrame(historia.history).to_csv(carpeta_resultados / 'historial.csv', index=False)

## 10. Métricas finales en imágenes de prueba

La matriz de confusión permite saber qué marcas confunde la red: por ejemplo, si interpreta un aspa como cruz o si no detecta la intersección fuera del cuadro.

In [ ]:
loss, acc_keras = model.evaluate(test_ds, verbose=0)
probs = model.predict(test_ds, verbose=1)  # model.predict como en clase
predichas = np.argmax(probs, axis=1)

resultados = {
    'accuracy': accuracy_score(y_test, predichas),
    'precision_macro': precision_score(y_test, predichas, average='macro', zero_division=0),
    'recall_macro': recall_score(y_test, predichas, average='macro', zero_division=0),
    'f1_macro': f1_score(y_test, predichas, average='macro', zero_division=0),
    'loss_test': loss,
    'muestras_test': len(y_test)
}
print(pd.Series(resultados))
reporte = classification_report(y_test, predichas, labels=range(len(clases)),
                                target_names=clases, output_dict=True, zero_division=0)
print(classification_report(y_test, predichas, labels=range(len(clases)),
                            target_names=clases, zero_division=0))
pd.Series(resultados, name='valor').to_csv(carpeta_resultados / 'metricas.csv')
pd.DataFrame(reporte).transpose().to_csv(carpeta_resultados / 'metricas_por_clase.csv')

cm = confusion_matrix(y_test, predichas, labels=range(len(clases)))
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=clases).plot(ax=ax, cmap='Blues', xticks_rotation=25)
plt.title('Matriz de confusión (prueba)')
plt.tight_layout()
plt.savefig(carpeta_resultados / 'matriz_confusion.png', dpi=150)
plt.show()

## 11. Probar una imagen individual

Repetimos el estilo de la clase: `load_img` → `img_to_array` → `expand_dims` → `preprocess_input` → `model.predict`.

In [ ]:
# Una imagen REAL del conjunto de prueba, no fabricada.
img_path = x_test[0]
img = image.load_img(img_path, target_size=target_size)
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)
preds = model.predict(x, verbose=0)
print(pd.DataFrame({'clase': clases, 'puntuacion': preds[0]}).sort_values('puntuacion', ascending=False))
print('Etiqueta real:', clases[y_test[0]])
plt.imshow(image.load_img(img_path)); plt.axis('off'); plt.show()

## 12. Guardar el modelo y sus nombres de clases

Descarga los archivos y colócalos juntos en la carpeta `models/` de la aplicación Flask. Sin ellos la aplicación puede recolectar dibujos, pero **no inventa predicciones**.

In [ ]:
# Igual orden que las salidas de softmax: índice 0,1,2,3.
model.save(carpeta_resultados / 'modelo_voto.keras')
(carpeta_resultados / 'classes.json').write_text(
    json.dumps(clases, ensure_ascii=False, indent=2), encoding='utf-8')
print('Guardado en:', carpeta_resultados)
print('Descarga modelo_voto.keras y classes.json, cópialos a models/ en tu PC.')
print('Puedes descargar también metricas.csv, matriz_confusion.png y curvas.png.')
# Para descargar desde Colab cuando termines:
# files.download(str(carpeta_resultados / 'modelo_voto.keras'))
# files.download(str(carpeta_resultados / 'classes.json'))

## 13. Conclusiones (se completarán luego)

**No redactar conclusiones sobre el rendimiento antes de entrenar.** Al finalizar pueden comentar:

- Cuántos dibujos manuales aportaron y cuántas personas participaron.
- Accuracy, precision, recall, F1 de la prueba y errores principales de la matriz de confusión.
- Qué tipo de marcas presenta más confusiones, por qué requieren revisión humana y qué limitaciones tiene el sistema.
- La utilidad educativa del reconocimiento de marcas con un canvas y los límites de usar este modelo en un contexto electoral.

La confianza mostrada por softmax no es certeza legal ni reemplaza la revisión humana.